# **ML : Assignment - 8 ( Q.4 )**

Mohd Talha Patrawala

CMPN - B

23102B0025

In [3]:
!pip install pandas numpy scikit-learn networkx scipy ucimlrepo

In [4]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from ucimlrepo import fetch_ucirepo

In [5]:
iris = fetch_ucirepo(id=53)

X = iris.data.features
y = iris.data.targets

print("Dataset loaded successfully")
print("Shape:", X.shape)

Dataset loaded successfully
Shape: (150, 4)


In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))

G = nx.Graph()
n = len(X_scaled)

for i in range(n):
    for j in range(i+1, n):
        G.add_edge(i, j, weight=dist_matrix[i][j])

print("Graph constructed")

Graph constructed


## Minimum Spanning Tree (MST)

The MST is constructed using Kruskal’s algorithm.  
It connects all nodes with minimum total edge weight without forming cycles.

In [8]:
mst = nx.minimum_spanning_tree(G, algorithm='kruskal')

print("MST constructed with edges:", len(mst.edges()))

MST constructed with edges: 149


In [9]:
k = 3

edges = sorted(mst.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)

mst_copy = mst.copy()
for i in range(k - 1):
    u, v, _ = edges[i]
    mst_copy.remove_edge(u, v)

components = list(nx.connected_components(mst_copy))

mst_labels = np.zeros(n)
for cluster_id, comp in enumerate(components):
    for node in comp:
        mst_labels[node] = cluster_id

# Save output
mst_output = pd.DataFrame({
    'SampleId': range(n),
    'ClusterLabel': mst_labels.astype(int)
})

mst_output.to_csv('mst_clusters.csv', index=False)

print("MST clustering done")

MST clustering done


In [10]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_output = pd.DataFrame({
    'SampleId': range(n),
    'ClusterLabel': kmeans_labels
})

kmeans_output.to_csv('kmeans_clusters.csv', index=False)

print("K-Means clustering done")

K-Means clustering done


In [11]:
mst_score = silhouette_score(X_scaled, mst_labels)
kmeans_score = silhouette_score(X_scaled, kmeans_labels)

print("Silhouette Score (MST):", mst_score)
print("Silhouette Score (K-Means):", kmeans_score)

Silhouette Score (MST): 0.5028765952374756
Silhouette Score (K-Means): 0.4589717867018717


**MST performs better when:**
- Clusters are non-spherical
- Data has irregular shapes

**K-Means performs better when:**
- Clusters are compact and well-separated
- Dataset size is large (computationally efficient)


---



# **Why MST Captures Non-Spherical Clusters**

Minimum Spanning Trees (MST) are highly effective at capturing non-spherical clusters because they rely on local connectivity rather than global centroids or cluster shapes. Unlike algorithms like K-means, which assume clusters are convex and hyperspherical, MST-based clustering (such as the single-linkage approach) focuses on the shortest edges between individual points.

This local focus allows the algorithm to follow high-density "paths" through the data, regardless of the overall geometry. Key reasons include:

* **Path-Based Connectivity:** It links points to their nearest neighbors, allowing the tree to "snake" through elongated, curved, or manifold structures.

* **Gap Sensitivity:** Clusters are separated by removing the longest edges (the "bottlenecks"), which naturally identifies natural voids between shapes.

* **No Centroid Bias:** Since it doesn't calculate a mean or center point, it isn't "pulled" toward the middle of a shape, preserving irregular boundaries.

* **Density Focus:** It treats dense regions as single components, effectively "ignoring" the distance between the ends of a long, thin cluster.